<a href="https://colab.research.google.com/github/Kaushikraviiyer/EDA-Coursework/blob/main/Lab_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import numpy as np
import pandas as pd

csv_file_path = "/content/fraudTest.csv"

print(f"--> Reading your uploaded file directly from: {csv_file_path}")

df = pd.read_csv(csv_file_path, low_memory=False)
print("\n--- [Step 1] Initial Raw Columns Available ---")
print(df.head(3))

if "amt" in df.columns:
    target_col = "amt"
elif "Amount" in df.columns:
    target_col = "Amount"
else:
    target_col = df.select_dtypes(include=[np.number]).columns[0]

print(f"\n--> Mapped analytical operations successfully onto column: '{target_col}'")

max_val = df[target_col].max() + 5000
risk_bins = [0, 50, 200, 1000, max_val]
risk_labels = ["Low Risk Tier", "Medium Risk Tier", "High Risk Tier", "Critical Risk/Flagged Fraud"]

df["FraudRiskCategory"] = pd.cut(df[target_col], bins=risk_bins, labels=risk_labels)

print("\n--- [Step 3] pd.cut Risk Distribution Tiers ---")
print(df["FraudRiskCategory"].value_counts())


df["SpendQuartile"] = pd.qcut(df[target_col], q=4, labels=["Q1_Low Spend", "Q2_Mid Spend", "Q3_High Spend", "Q4_Top Spend"])

print("\n--- [Step 4] pd.qcut Quantile Balance Inspection ---")
print(df["SpendQuartile"].value_counts())


outlier_threshold = 1200
outlier_mask = np.abs(df[target_col]) > outlier_threshold

suspected_fraud_df = df[outlier_mask]

print(f"\n--- [Step 5] Slicing Outlier Discrepancies (> ${outlier_threshold}) ---")
print(f"Total anomaly transactions extracted from dataset: {len(suspected_fraud_df)}")

if not suspected_fraud_df.empty:

    display_cols = ["trans_date_trans_time", "cc_num", "merchant", "category", target_col, "FraudRiskCategory"]
    valid_display = [c for c in display_cols if c in df.columns]

    print("\nSample of Extracted Outlier Discrepancies:")
    print(suspected_fraud_df[valid_display].head(10))
else:
    print("Zero records crossed the specified anomaly limit.")


--> Reading your uploaded file directly from: /content/fraudTest.csv

--- [Step 1] Initial Raw Columns Available ---
   Unnamed: 0 trans_date_trans_time            cc_num  \
0           0   2020-06-21 12:14:25  2291163933867244   
1           1   2020-06-21 12:14:33  3573030041201292   
2           2   2020-06-21 12:14:53  3598215285024754   

                               merchant        category    amt   first  \
0                 fraud_Kirlin and Sons   personal_care   2.86    Jeff   
1                  fraud_Sporer-Keebler   personal_care  29.84  Joanne   
2  fraud_Swaniawski, Nitzsche and Welch  health_fitness  41.28  Ashley   

       last gender                street  ...      lat      long  city_pop  \
0   Elliott      M     351 Darlene Green  ...  33.9659  -80.9355    333497   
1  Williams      F      3638 Marsh Union  ...  40.3207 -110.4360       302   
2     Lopez      F  9333 Valentine Point  ...  40.6729  -73.5365     34496   

                      job         dob       